# Dataproc + BigQuery Connector Patterns — Demo

Companion notebook to the **Dataproc + BigQuery Connector Patterns Handbook**.

This notebook submits a Dataproc Serverless batch job that reads the `hr_demo.employees` table (from earlier handbooks) into a Spark DataFrame, transforms it, and writes the result to BigQuery **twice** — once using the INDIRECT write method and once using DIRECT — so you can compare them side by side: job history, latency, and cost characteristics.

Run the cells top to bottom. Update the variables in the **Configuration** cell first.


## 0. Setup: install & authenticate


In [ ]:
!pip install --quiet google-cloud-dataproc google-cloud-bigquery google-cloud-storage


In [ ]:
try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab.')
except ImportError:
    print('Not running in Colab -- make sure you have run `gcloud auth application-default login` locally.')


## 1. Configuration — edit before running


In [ ]:
PROJECT_ID = "project-2df9d2ad-b6f5-4ed5-997"
REGION = "us-central1"
TEMP_BUCKET = f"{PROJECT_ID}-spark-temp"
SOURCE_TABLE = f"{PROJECT_ID}.hr_demo.employees"
INDIRECT_TABLE = f"{PROJECT_ID}.hr_demo.employees_indirect"
DIRECT_TABLE = f"{PROJECT_ID}.hr_demo.employees_direct"

print(f"Project: {PROJECT_ID} | Region: {REGION} | Temp bucket: {TEMP_BUCKET}")


## 2. Create the temporary GCS staging bucket (used by INDIRECT writes)


In [ ]:
from google.cloud import storage

storage_client = storage.Client(project=PROJECT_ID)

try:
    bucket = storage_client.create_bucket(TEMP_BUCKET, location=REGION)
    print(f"Bucket created: {bucket.name}")
except Exception as e:
    print(f"Bucket may already exist: {e}")


## 3. Grant the batch job's service account the roles it needs

Covers both read (Data Viewer + Read Session User) and write (Data Editor + Job User + Storage Object Admin for staging).


In [ ]:
import google.auth
import google.auth.transport.requests
import requests

creds, _ = google.auth.default()
creds.refresh(google.auth.transport.requests.Request())
headers = {
    "Authorization": f"Bearer {creds.token}",
    "Content-Type": "application/json",
}

project_url = f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}"
project_info = requests.get(project_url, headers=headers).json()
project_number = project_info["projectNumber"]
service_account = f"{project_number}[email protected]"
print(f"Service account: {service_account}")

get_policy = requests.post(f"{project_url}:getIamPolicy", headers=headers, json={}).json()
bindings = get_policy.get("bindings", [])

needed_roles = [
    "roles/bigquery.dataEditor",
    "roles/bigquery.dataViewer",
    "roles/bigquery.jobUser",
    "roles/bigquery.readSessionUser",
    "roles/storage.objectAdmin",
]

for role in needed_roles:
    binding = next((b for b in bindings if b["role"] == role), None)
    if binding is None:
        binding = {"role": role, "members": []}
        bindings.append(binding)
    member = f"serviceAccount:{service_account}"
    if member not in binding["members"]:
        binding["members"].append(member)

get_policy["bindings"] = bindings
set_resp = requests.post(f"{project_url}:setIamPolicy", headers=headers, json={"policy": get_policy})
print(set_resp.status_code, "-- IAM roles granted (or already present).")


## 4. Generate the PySpark script


In [ ]:
pyspark_script = f"""
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper

spark = SparkSession.builder.appName("ConnectorPatternsDemo").getOrCreate()

# --- READ: direct table read with pushdown ---
df = spark.read.format("bigquery") \\
    .option("table", "{SOURCE_TABLE}") \\
    .load()

transformed = df.filter(col("department") == "Engineering") \\
                .withColumn("department_upper", upper(col("department")))

transformed.show()

# --- WRITE (INDIRECT): staged via GCS, loaded via BQ load job ---
transformed.write.format("bigquery") \\
    .option("table", "{INDIRECT_TABLE}") \\
    .option("writeMethod", "indirect") \\
    .option("temporaryGcsBucket", "{TEMP_BUCKET}") \\
    .mode("overwrite") \\
    .save()

# --- WRITE (DIRECT): straight via Storage Write API ---
transformed.write.format("bigquery") \\
    .option("table", "{DIRECT_TABLE}") \\
    .option("writeMethod", "direct") \\
    .mode("overwrite") \\
    .save()

print("Done. Compare {INDIRECT_TABLE} vs {DIRECT_TABLE} in BigQuery.")
"""

with open("connector_demo.py", "w") as f:
    f.write(pyspark_script)

print("Script written to connector_demo.py")
print(pyspark_script)


In [ ]:
bucket_obj = storage_client.bucket(TEMP_BUCKET)
blob = bucket_obj.blob("scripts/connector_demo.py")
blob.upload_from_filename("connector_demo.py")
SCRIPT_URI = f"gs://{TEMP_BUCKET}/scripts/connector_demo.py"
print(f"Uploaded script to {SCRIPT_URI}")


In [ ]:
from google.cloud import bigquery

bq_client = bigquery.Client(project=PROJECT_ID)

# Create the dataset
dataset_ref = bigquery.DatasetReference(PROJECT_ID, "hr_demo")
dataset = bigquery.Dataset(dataset_ref)
dataset.location = "us"
bq_client.create_dataset(dataset, exists_ok=True)

# Create the table
schema = [
    bigquery.SchemaField("employee_id", "INT64"),
    bigquery.SchemaField("full_name", "STRING"),
    bigquery.SchemaField("department", "STRING"),
    bigquery.SchemaField("email", "STRING"),
    bigquery.SchemaField("salary", "NUMERIC"),
]
table_ref = bigquery.TableReference(dataset_ref, "employees")
table = bigquery.Table(table_ref, schema=schema)
bq_client.create_table(table, exists_ok=True)

# Insert sample rows
rows = [
    {"employee_id": 1, "full_name": "Riya Sharma", "department": "Engineering", "email": "[email protected]", "salary": 950000},
    {"employee_id": 2, "full_name": "Arjun Verma", "department": "Sales", "email": "[email protected]", "salary": 720000},
    {"employee_id": 3, "full_name": "Meera Nair", "department": "Finance", "email": "[email protected]", "salary": 880000},
]
errors = bq_client.insert_rows_json(table, rows)
print("Insert errors:", errors if errors else "none")

## 5. Submit the batch job


In [ ]:
from google.cloud import dataproc_v1

batch_client = dataproc_v1.BatchControllerClient(
    client_options={"api_endpoint": f"{REGION}-dataproc.googleapis.com:443"}
)

batch = dataproc_v1.Batch()
batch.pyspark_batch.main_python_file_uri = SCRIPT_URI
batch.runtime_config.version = "2.2"

parent = f"projects/{PROJECT_ID}/locations/{REGION}"
operation = batch_client.create_batch(parent=parent, batch=batch)
print("Batch submitted. Waiting for it to complete (this can take a few minutes)...")

result = operation.result()  # blocks until the batch finishes
print("Batch finished with state:", result.state)
batch_id = result.name.split("/")[-1]
print(f"View logs: gcloud dataproc batches describe {batch_id} --region={REGION}")


## 6. Compare the two output tables

Row counts should match; the interesting difference is in *how* each was written — check the BigQuery Jobs Explorer for a visible LOAD job on the indirect table and its absence for the direct one.


In [ ]:
from google.cloud import bigquery

bq_client = bigquery.Client(project=PROJECT_ID)

for label, table in [("INDIRECT", INDIRECT_TABLE), ("DIRECT", DIRECT_TABLE)]:
    try:
        t = bq_client.get_table(table)
        print(f"{label:10s} | rows={t.num_rows:5d} | size_bytes={t.num_bytes:8d} | created={t.created}")
    except Exception as e:
        print(f"{label:10s} | Error: {e}")


## 7. Cleanup


In [ ]:
CONFIRM_CLEANUP = True  # flip to True to actually delete resources

if CONFIRM_CLEANUP:
    bq_client.delete_table(INDIRECT_TABLE, not_found_ok=True)
    bq_client.delete_table(DIRECT_TABLE, not_found_ok=True)
    blobs = list(bucket_obj.list_blobs())
    for b in blobs:
        b.delete()
    bucket_obj.delete()
    print("Output tables and staging bucket deleted.")
else:
    print("Skipped. Set CONFIRM_CLEANUP = True and re-run this cell to clean up.")
